#### Imports

In [1]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
import struct
import math
from tqdm import tqdm
import time
import random
from sklearn.cluster import KMeans
import warnings

#### Useful functions

In [2]:
### Reads binary files in input_dir for the levels between min_level and max_level,
### and returns them in a list of numpy arrays, one for each chunk of data (box).
### Also returns a list of the tuples of the location and dimension of each
### box. Finally, returns a list of the number of boxes at each level.
def process_data(dir_prefix, timestep, min_level, max_level, components):
    
    # create output lists
    boxes, locations, dimensions, box_counts = [], [], [], []

    print("Processing data...")
    for l in tqdm(range(min_level, max_level+1)): # iterate over each level
        filenames = []
        for component in components:
            filename = f"{dir_prefix}-{component}-{l}/{timestep}-wholeNewFormat-{component}-{l}.raw"
            filenames.append(filename)

        # Keeps track of how many boxes are at each level
        box_count = 0
        boxes_l, locations_l, dimensions_l = [], [], []

        # First round (store loc/dim, create boxes)
        with open(filenames[0], "rb") as file:
            while True: # Iterate through all the data at the current level until none is left
                test = file.read(4)
                if len(test) < 4: # Break the loop if all data has been read
                    break
                # Read the location of the box
                x = int(struct.unpack('<f', test)[0])
                y = int(struct.unpack('<f', file.read(4))[0])
                z = int(struct.unpack('<f', file.read(4))[0])
                locations_l.append((x, y, z))

                # Read the dimensions of the box
                xdim = int(struct.unpack('<f', file.read(4))[0])
                ydim = int(struct.unpack('<f', file.read(4))[0])
                zdim = int(struct.unpack('<f', file.read(4))[0])
                dimensions_l.append((xdim, ydim, zdim))

                # Read the data in the box
                box = np.empty((xdim, ydim, zdim, len(components)), dtype=np.float32)
                for k in range(zdim):
                    for j in range(ydim):
                        for i in range(xdim):
                            box[i,j,k,0] = struct.unpack('<f', file.read(4))[0]
                boxes_l.append(box)
                box_count += 1

        # Second round (add other components)
        for idx, filename in enumerate(filenames[1:]):
            with open(filenames[idx+1], "rb") as file:
                for box_idx in range(len(boxes_l)):
                    
                    # Read the location of the box
                    x = int(struct.unpack('<f', file.read(4))[0])
                    y = int(struct.unpack('<f', file.read(4))[0])
                    z = int(struct.unpack('<f', file.read(4))[0])

                    # Read the dimensions of the box
                    xdim = int(struct.unpack('<f', file.read(4))[0])
                    ydim = int(struct.unpack('<f', file.read(4))[0])
                    zdim = int(struct.unpack('<f', file.read(4))[0])

                    for k in range(zdim):
                        for j in range(ydim):
                            for i in range(xdim):
                                boxes_l[box_idx][i,j,k,idx+1] = struct.unpack('<f', file.read(4))[0]

        boxes.append(boxes_l)
        locations.append(locations_l)
        dimensions.append(dimensions_l)
        box_counts.append(box_count)
        
    return boxes, locations, dimensions, box_counts



### Takes a box and splits it into cubes (blocks) of a specified dimension. Returns
### a list of numpy arrays, each representing one block of the box. If block_dim
### does not divide equally into the volume dimensions, creates blocks that extend past the
### dimensions of the volume, and populates the coordinates outside the volume with chosen 
### quantity junk, which is an input to the function
def to_blocks(box, block_dim, junk, components):
    
    xdim = box.shape[0]
    ydim = box.shape[1]
    zdim = box.shape[2]

    x_steps = int(xdim // block_dim)
    if xdim % block_dim != 0:
        x_steps += 1
    if xdim < block_dim:
        x_steps = 1
    y_steps = int(ydim / block_dim)
    if ydim % block_dim != 0:
        y_steps += 1
    if ydim < block_dim:
        y_steps = 1
    z_steps = int(zdim / block_dim)
    if zdim % block_dim != 0:
        z_steps += 1
    if zdim < block_dim:
        z_steps = 1

    blocks = []
    
    for k in range(z_steps):
        for j in range(y_steps):
            for i in range(x_steps):

                x_lo, y_lo, z_lo = int(i * block_dim), int(j * block_dim), int(k * block_dim)
                x_hi, y_hi, z_hi = min(x_lo + block_dim, xdim), min(y_lo + block_dim, ydim), min(z_lo + block_dim, zdim)

                # create a block, fill with junk
                block = np.full((block_dim, block_dim, block_dim, len(components)), junk, dtype=np.float64)

                # slice the block from the original box
                block[:x_hi - x_lo, :y_hi - y_lo, :z_hi - z_lo, :] = box[x_lo:x_hi, y_lo:y_hi, z_lo:z_hi, :]

                block = block.reshape(-1, len(components))
                blocks.append(block)

    return blocks



### Takes a set of read boxes, and breaks them into blocks of a specified
### dimension for use in creating a codebook.
def create_samples(boxes, block_dim, junk, components):

    print("Creating samples...")
    
    # Check if block_dim is a power of 2
    if not math.log2(block_dim).is_integer():
        print("Invalid block_dim! Dimension must be a power of 2.")
        return
    
    samples = []

    for l in tqdm(range(len(boxes))): # iterate through all levels
        boxes_l = boxes[l]
        samples_l = []
    
        for box in boxes_l: # Iterate through each box at the current level
            # Create blocks of the desired dimension from the box
            blocks = to_blocks(box, block_dim, junk, components)
            samples_l.append(blocks)

        samples.append(samples_l)

    return samples


### Finds num_modes modes for a set of values using KMeans clustering. If there are
### fewer distinct values than num_modes, returns one of the distinct values for all
### modes.
def cluster_data(values, num_modes):

    # Convert values to a numpy array and reshape for KMeans
    values = cp.array(values).reshape(-1, 1)

    # Catch the ConvergenceWarning and handle it differently
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)  # Catch all warnings

        try:
            # Apply KMeans clustering
            kmeans = KMeans(n_clusters=num_modes, random_state=42)
            kmeans.fit(values.get())
    
            # Get the cluster centers (modes) and assign each value to a cluster
            modes = sorted(kmeans.cluster_centers_.flatten())
    
        except UserWarning as e:
            if "Number of distinct clusters" in str(e):
                # In case of fewer distinct clusters, fall back to using the single unique value
                modes = []
                for n in range(num_modes):
                    modes.append(float(np.unique(values.get())[0]))
            
    return modes


### The original approach: takes two endpoints, and finds num_intermediate evenly
### spaced points between them
def generate_intermediate_data(data1, data2, num_intermediate):

    interpolation_factors = cp.linspace(0, 1, num_intermediate + 2)

    points = [(1 - t) * data1 + t * data2 for t in interpolation_factors]

    return points



### Compresses a given block using a BCn like framework into a palette and indices.
def compress_block(block, origin, components, num_intermediate, block_dim):

    block = np.array(block) 
    # distances = np.linalg.norm(block - origin, axis=1)

    # min_pix = np.argmin(distances)
    # max_pix = np.argmax(distances)

    palette = cp.zeros((num_intermediate, len(components)))
    
    # Create palette
    for c in range(len(components)):
        modes = cluster_data(block[:, c], num_intermediate)
        palette[:, c] = cp.array(modes)
        
    # palette = cp.array(generate_intermediate_data(block[min_pix], block[max_pix], num_intermediate))
    block = cp.array(block)

    block_expanded = cp.expand_dims(block, axis=1)
    palette_expanded = cp.expand_dims(palette, axis=0)
    distances_to_palette = cp.linalg.norm(block_expanded - palette_expanded, axis=2)

    indices = cp.argmin(distances_to_palette, axis=1, out=cp.zeros((block_dim**3), dtype=cp.uint8))
    
    # indices = cp.zeros((block_dim**3), dtype=cp.uint8)
    # for i in range(block_dim**3):
    #     pix = block[i]
    #     min_distance = float('inf')
    #     min_index = 0
    #     for idx, value in enumerate(palette):
    #         distance = cp.linalg.norm(cp.array(pix) - cp.array(value)) # Euclidian distance
    #         if distance < min_distance:
    #             min_distance = distance
    #             min_index = idx
    #     indices[i] = min_index

    return palette, indices

    

### Reconstructs a box with dimensions box_dim from BCn compressed format
def reconstruct_box(box_dim, block_dim, indices, palettes, components):

    # Extract the dimensions of the box
    xdim, ydim, zdim = box_dim[0], box_dim[1], box_dim[2]

    # Create an empty box to store the reconstructed data
    reconstructed = cp.zeros((xdim, ydim, zdim, len(components)))

    x_steps = int(xdim // block_dim)
    if xdim % block_dim != 0:
        x_steps += 1
    if xdim < block_dim:
        x_steps = 1
    y_steps = int(ydim / block_dim)
    if ydim % block_dim != 0:
        y_steps += 1
    if ydim < block_dim:
        y_steps = 1
    z_steps = int(zdim / block_dim)
    if zdim % block_dim != 0:
        z_steps += 1
    if zdim < block_dim:
        z_steps = 1

    block_idx = 0
    
    for k in range(z_steps):
        for j in range(y_steps):
            for i in range(x_steps):

                start = time.time()
                reconstructed_block = cp.zeros((block_dim**3, len(components)))

                palette = palettes[block_idx]
                #print("Palette generation:", end - start, "seconds")

                start = time.time()
                # palette = cp.array(palette)
                for n in range(block_dim**3):
                    reconstructed_block[n] = palette[block_indices[n]]
                # for p in range(block_dim**3):
                #     palette_idx = int(block_indices[p])
                #     reconstructed_block[p] = palette[palette_idx]
                end = time.time()
                #print("Block filling:", end - start, "seconds")

                start = time.time()
                reshaped_block = reconstructed_block.reshape(block_dim, block_dim, block_dim, -1)
                end = time.time()
                #print("Reshaping:", end - start, "seconds")

                start = time.time()
                x_lo, y_lo, z_lo = int(i * block_dim), int(j * block_dim), int(k * block_dim)
                x_hi, y_hi, z_hi = min(x_lo + block_dim, xdim), min(y_lo + block_dim, ydim), min(z_lo + block_dim, zdim)
                reconstructed[x_lo:x_hi, y_lo:y_hi, z_lo:z_hi, :] = reshaped_block[:x_hi - x_lo, :y_hi - y_lo, :z_hi - z_lo, :]
                end = time.time()
                #print("Slicing:", end - start, "seconds")

                block_idx += 1

    return reconstructed


### Calculates the average root mean squared error for a given list of boxes
### and their regenerations
def calc_avg_rmse(actuals, regens, components):

    num_components = len(components)

    global_rmses = [None] * num_components

    for c in range(num_components): # iterate over components

        rmses_c = []
        
        for l in range(len(actuals)): # iterate over levels
            boxes_l = actuals[l]
            regen_boxes_l = regens[l]
    
            for box_idx in tqdm(range(len(boxes_l))): # iterate through boxes at current level

                actual = boxes_l[box_idx]
                pred = regen_boxes_l[box_idx]
        
                xdim = actual.shape[0]
                ydim = actual.shape[1]
                zdim = actual.shape[2]

                sum = 0
        
                for k in range(zdim):
                    for j in range(ydim):
                        for i in range(xdim):
                            sq = (actual[i][j][k][c] - pred[i][j][k][c])**2
                            sum += sq

                rmse = (sum / (xdim * ydim * zdim))**0.5
            
                rmses_c.append(rmse)
    
        total = 0

        for n in range(len(rmses_c)):
            total += rmses_c[n]
        
        global_rmses[c] = total / len(rmses_c)

    return global_rmses

#### Hyperparameters

In [9]:
dir_prefix = "wholeVolumesNewFormat"
components = [6, 25, 37, 46, 55]
timestep = 74
min_level = 0
max_level = 0
block_dim = 8
origin = cp.zeros((len(components)), dtype=cp.float32)
junk = 0
num_intermediate = 16

#### Data preprocessing

In [10]:
boxes, locations, dimensions, box_counts = process_data(dir_prefix, timestep, min_level, max_level, components)

blocks = create_samples(boxes, block_dim, junk, components)

Processing data...


100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.82s/it]


Creating samples...


100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 21.12it/s]


#### Compression

In [11]:
# min_pixes = []
# max_pixes = []
palettes = []
indices = []

for lev in range(min_level, max_level+1):
    # min_pixes_l = []
    # max_pixes_l = []
    palettes_l = []
    indices_l = []
    for box_idx in tqdm(range(len(blocks[lev]))):
        # box_min_pixes = []
        # box_max_pixes = []
        box_palettes = []
        box_indices = []
        for block_idx in range(len(blocks[lev][box_idx])):
            block_palette, block_indices = compress_block(blocks[lev][box_idx][block_idx], 
                                                       origin, components, num_intermediate, block_dim)
            # box_min_pixes.append(min_pix)
            # box_max_pixes.append(max_pix)
            box_indices.append(block_indices)
            box_palettes.append(block_palette)

        # min_pixes_l.append(box_min_pixes)
        # max_pixes_l.append(box_max_pixes)
        indices_l.append(box_indices)
        palettes_l.append(box_palettes)

    # min_pixes.append(min_pixes_l)
    # max_pixes.append(max_pixes_l)
    indices.append(indices_l)
    palettes.append(palettes_l)

100%|█████████████████████████████████████████████████████████████████████████████████| 576/576 [01:10<00:00,  8.14it/s]


#### Check size of compressed data

In [6]:
# Check size of indices/palettes
indices_size = 0
palettes_size = 0
for lev in range(min_level, max_level+1):
    for box_idx in range(len(indices[lev])):
        for block_idx in range(len(indices[lev][box_idx])):
            indices_size += indices[lev][box_idx][block_idx].nbytes
            palettes_size += palettes[lev][box_idx][block_idx].nbytes
print("Indices size:", indices_size, "bytes")
print("Palettes size:", palettes_size, "bytes")

# # Check size of loc/dim data
# locations_size = 0
# dimensions_size = 0
# for t in range(len(locations)):
#     for lev in range(len(locations[time])):
#         for box_idx in range(len(locations[time][lev])):
#             for i in range(3):
#                 locations_size += 4
#                 dimensions_size += 4
# print("Location/dimension data size:", locations_size + dimensions_size, "bytes")

Indices size: 2359296 bytes
Palettes size: 23592960 bytes


#### Decompress timestep

In [12]:
regen_boxes = []

for l in range(min_level, max_level+1): # iterating over level

    dimensions_l = dimensions[l]
    # min_pixes_l = min_pixes[l]
    # max_pixes_l = max_pixes[l]
    indices_l = indices[l]
    palettes_l = palettes[l]
    regen_boxes_l = [None] * len(dimensions_l)

    for box_idx in tqdm(range(len(dimensions_l))):
        regen_boxes_l[box_idx] = reconstruct_box(dimensions_l[box_idx], block_dim, indices_l[box_idx], palettes_l[box_idx],
                                                components)

    regen_boxes.append(regen_boxes_l)

rmses = calc_avg_rmse(boxes, regen_boxes, components)
for i in range(len(rmses)):
    print(rmses[i])

100%|█████████████████████████████████████████████████████████████████████████████████| 576/576 [02:33<00:00,  3.74it/s]

178.8571921587069
7370389.085945567
0.011752200994255023
1259.330065650423
0.22553621578705282
